# 05 - Target Distribution & Class Imbalance

## Context
Ketidakseimbangan kelas (class imbalance) adalah karakteristik utama dalam kasus deteksi fraud. Dengan hanya sekitar 3.5% transaksi fraud, model yang naif hanya akan memprediksi "bukan fraud" untuk semua transaksi dan mendapatkan akurasi 96.5%, tapi sama sekali tidak berguna dalam praktiknya.

## Objective
- Menghitung rasio ketidakseimbangan kelas secara tepat.
- Menganalisis sebaran fraud pada beberapa fitur penting.
- Menghitung bobot kelas untuk strategi modeling.

### Impor Library & Load Data

In [ ]:
# Impor library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Muat data hasil merge
train = pd.read_parquet('../data/interim/train_merged.parquet')
print(f'data loaded :{train.shape}')

### Eksplorasi: Distribusi Target (`isFraud`)

In [ ]:
# Hitung sebaran kelas target
fraud_counts = train['isFraud'].value_counts()
fraud_pct = train['isFraud'].value_counts(normalize=True) * 100

print('Target Distribution:')
print(f'Not Fraud = {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)')
print(f'Fraud     = {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)')
print(f'Imbalance Ratio : {fraud_counts[0] // fraud_counts[1]}:1')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot bar jumlah transaksi
axes[0].bar(['Not Fraud', 'Fraud'],
            fraud_counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Distribution (Count)')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 10000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart persentase
axes[1].pie(fraud_pct.values,
            labels=['Not Fraud', 'Fraud'],
            colors=['#2ecc71', '#e74c3c'],
            autopct='%1.2f%%',
            startangle=90,
            explode=[0, 0.15]
            )
axes[1].set_title('Percentage of Target Distribution')
plt.tight_layout()
plt.show()

### Eksplorasi: Fraud berdasarkan Kode Produk (`ProductCD`)

In [ ]:
# Analisis fraud rate untuk tiap ProductCD
product_fraud = train.groupby('ProductCD')['isFraud'].agg(['count','sum','mean'])
product_fraud.columns = ['Count', 'Fraud Count', 'Fraud Rate']
product_fraud['Fraud Rate'] = product_fraud['Fraud Rate'] * 100

print('Fraud Rate by Product Code:')
print(product_fraud.sort_values('Fraud Rate', ascending=False).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
product_fraud['Fraud Rate'].plot(kind='bar', ax=ax, color='coral')
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate By Product Code')
ax.axhline(y=train['isFraud'].mean()*100, color='red', linestyle='--', label='Overall Average')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Eksplorasi: Fraud berdasarkan Nominal Transaksi (`TransactionAmt`)

In [ ]:
# Bagi nominal transaksi jadi 10 bagian sama besar (decile)
train['amt_bin'] = pd.qcut(train['TransactionAmt'],
                           q=10,
                           duplicates='drop')
amt_fraud = train.groupby('amt_bin', observed=False)['isFraud'].agg(['count','mean'])
amt_fraud.columns = ['Count', 'Fraud Rate']
amt_fraud['Fraud Rate'] = amt_fraud['Fraud Rate']*100

print('Fraud rate by Transaction Amount')
print(amt_fraud.to_string())

fig, ax = plt.subplots(figsize=(10,6))
amt_fraud['Fraud Rate'].plot(kind='bar', ax= ax , color = 'blue')
ax.set_title('Fraud By Transaction Amt')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Amount Bin')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### Perhitungan Bobot Kelas (Class Weighting)

In [ ]:
# Hitung rekomendasi bobot kelas untuk model xgboost / lightgbm
n_neg = (train['isFraud'] == 0).sum()
n_pos = (train['isFraud'] == 1).sum()
scale_pos_weight = n_neg / n_pos

print('Class Weighting Strategy:')
print(f'Negative Samples : {n_neg:,}')
print(f'Positive Samples : {n_pos:,}')
print(f'Recommended scale_pos_weight : {scale_pos_weight:.4f}')
print('\nUsage in LightGBM/XGBoost:')
print(f"params['scale_pos_weight'] = {scale_pos_weight:.0f}")